In [1]:
import warnings

import skrub
from sklearn.ensemble import HistGradientBoostingClassifier
from skrub import TableVectorizer

import sempipes

warnings.filterwarnings("ignore")


dataset = skrub.datasets.fetch_midwest_survey()


responses = skrub.var("responses")
responses = responses.skb.set_description(dataset.metadata["description"])

labels = skrub.var("labels")
labels = labels.skb.set_description(dataset.metadata["target"])

responses = responses.skb.mark_as_X()
labels = labels.skb.mark_as_y()

with_demographic_features = responses.sem_gen_features(
    nl_prompt="""
        Compute additional demographics-related features, use your intrinsic knowledge about the US. 
        Take into account how the identification with the country or regions of it changed over the generations.         
        Also think about how the identification differs per class and education. The midwest is generally associated 
        with "Midwestern values" — friendliness, modesty, hard work, and community-mindedness.
    """,
    name="demographic_features",
    how_many=5,
)

with_more_demographic_features = with_demographic_features.sem_gen_features(
    nl_prompt="""
        Compute even more demographics-related features!
    """,
    name="more_demographic_features",
    how_many=5,
)

with_spatial_features = responses.sem_gen_features(
    nl_prompt="""
        Compute additional geo-spatial-related features, use your intrinsic knowledge about the US.
    """,
    name="spatial_features",
    how_many=5,
)

all_features = with_more_demographic_features.skb.concat([with_spatial_features], axis=1)

feature_encoder = TableVectorizer()
encoded_responses = all_features.skb.apply(feature_encoder)
pipeline = encoded_responses.skb.apply(HistGradientBoostingClassifier(), y=labels)

In [2]:
pipeline

<Apply HistGradientBoostingClassifier>

In [3]:
from sempipes.optimisers import EvolutionarySearch, MonteCarloTreeSearch, TreeSearch, optimise_colopro

tuning_data = {"responses": dataset.X.head(n=500), "labels": dataset.y.head(n=500)}

outcomes = optimise_colopro(
    pipeline,
    num_trials=16,
    scoring="accuracy",
    search=MonteCarloTreeSearch(),  # TreeSearch(),#EvolutionarySearch(population_size=6),
    cv=2,
    additional_env_variables=tuning_data,
)

2026-02-26 11:47:45,777 - INFO - SEMPIPES> COLOPRO> Computing pipeline summary for context-aware optimisation
2026-02-26 11:47:47,186 - INFO - SEMPIPES> COLOPRO> Processing trial 0
2026-02-26 11:47:47,217 - INFO - SEMPIPES> COLOPRO> Initialising optimisation via OPRO
2026-02-26 11:47:47,238 - INFO - SEMPIPES> COLOPRO> Creating root node
2026-02-26 11:47:47,239 - INFO - SEMPIPES> COLOPRO> Scoring pipeline via 2-fold cross-validation
2026-02-26 11:47:59,531 - INFO - SEMPIPES> COLOPRO> Pipeline scoring took 12.29 seconds
2026-02-26 11:47:59,533 - INFO - SEMPIPES> COLOPRO> Score changed from None to 0.792
2026-02-26 11:47:59,534 - INFO - SEMPIPES> COLOPRO> Processing trial 1
2026-02-26 11:47:59,588 - INFO - SEMPIPES> MCT_SEARCH> Expanding childless node 0
2026-02-26 11:47:59,589 - INFO - SEMPIPES> MCT_SEARCH> Trying to improve node with score 0.792
2026-02-26 11:47:59,590 - INFO - SEMPIPES> COLOPRO> Trying to improve node with score 0.792
2026-02-26 11:47:59,590 - INFO - SEMPIPES> COLOPRO>

In [4]:
best_outcome = max(outcomes, key=lambda x: x.score)
best_outcome.score

0.874

In [5]:
from IPython.display import Code, display

display(Code(best_outcome.states.get("demographic_features")["generated_code"]))

import pandas as pd
import numpy as np

def _sem_gen_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Generates additional demographic and identification-related features for the dataset.

    Args:
        df (pd.DataFrame): The input DataFrame.

    Returns:
        pd.DataFrame: The DataFrame with new feature columns added.
    """

    # Feature 1: zip_code_region_prefix
    # This feature extracts the first digit of the ZIP code, which often corresponds to a broad geographical region
    # in the US. This provides a high-level demographic and geographical context for the respondent's location,
    # which is highly relevant for predicting their Census_Region. The previous code's fix for non-numeric
    # first characters is retained for robustness.
    # Input samples: 'In_what_ZIP_code_is_your_home_located': ['74070', '44106', '48185']
    df['zip_code_region_prefix'] = df['In_what_ZIP_code_is_your_home_located'].astype(str).str[0]
    df['zip_code_region_prefix'] = pd.to_numeric(df['zip_code_region_prefix'], errors='coerce')

    # Feature 2: personal_midwestern_identity_score
    # This feature converts the ordinal categorical variable of personal identification into a numerical score.
    # This allows the model to directly use the strength of a respondent's self-identification as a Midwesterner,
    # which is a core aspect of the survey and directly relates to the target.
    # Input samples: 'How_much_do_you_personally_identify_as_a_Midwesterner': ['Not much', 'Not much', 'A lot']
    identity_mapping = {
        'Not at all': 0,
        'Not much': 1,
        'Some': 2,
        'A lot': 3
    }
    df['personal_midwestern_identity_score'] = df['How_much_do_you_personally_identify_as_a_Midwesterner'].map(identity_mapping)

    # Feature 3: age_group_numeric
    # This feature converts age ranges into an ordinal numerical representation. This helps capture generational
    # differences in how people identify with regions. Younger generations might have different perceptions or
    # stronger/weaker ties to regional identities compared to older generations, reflecting changes over time.
    # Input samples: 'Age': ['18-29', '18-29', '30-44']
    age_mapping = {
        '18-29': 0,
        '30-44': 1,
        '45-60': 2,
        '60+': 3
    }
    df['age_group_numeric'] = df['Age'].map(age_mapping)

    # Feature 4: household_income_numeric
    # This feature converts household income ranges into an ordinal numerical representation. This directly
    # addresses the prompt's request to consider how identification differs per "class". Income is a key
    # demographic indicator that can influence regional identity, values, and social mobility.
    # Input samples: 'Household_Income': ['$50,000 - $99,999', '$0 - $24,999', '$25,000 - $49,999']
    income_mapping = {
        '$0 - $24,999': 0,
        '$25,000 - $49,999': 1,
        '$50,000 - $99,999': 2,
        '$100,000 - $149,999': 3,
        '$150,000+': 4
    }
    df['household_income_numeric'] = df['Household_Income'].map(income_mapping)

    # Feature 5: midwest_self_identification_consistency
    # This binary feature indicates whether a respondent's free-text description of their current region
    # (`What_would_you_call_the_part_of_the_country_you_live_in_now`) explicitly contains "Midwest" (case-insensitive).
    # This captures the consistency between where they perceive themselves to live and the region of interest.
    # A 'True' value suggests a direct alignment and potentially a stronger sense of belonging or "community-mindedness"
    # with the Midwest, which is relevant to "Midwestern values".
    # Input samples: 'What_would_you_call_the_part_of_the_country_you_live_in_now': ['Southern', 'Midwest', 'Mid-west']
    df['midwest_self_identification_consistency'] = df['What_would_you_call_the_part_of_the_country_you_live_in_now'].str.contains(
        'midwest|mid-west|mid west|upper midwest', case=False, na=False
    ).astype(int) # Convert b

In [6]:
display(Code(best_outcome.states.get("more_demographic_features")["generated_code"]))

import pandas as pd
import numpy as np

def _sem_gen_features(df: pd.DataFrame) -> pd.DataFrame:
    df_copy = df.copy()

    # Helper mapping for 'Yes'/'No' columns
    yes_no_map = {'Yes': 1, 'No': 0}

    # (gender_numeric: Binary encoding of Gender)
    # Input samples: 'Gender': ['Male', 'Male', 'Male']
    # Rationale: Gender is a fundamental demographic attribute that can influence regional identity and perception.
    # Encoding it numerically allows the downstream model to directly utilize this information, potentially
    # revealing gender-specific patterns in regional identification.
    df_copy['gender_numeric'] = df_copy['Gender'].map({'Male': 0, 'Female': 1})

    # (self_reported_midwest_flag: Binary flag if respondent self-identifies their current region as 'Midwest')
    # Input samples: 'What_would_you_call_the_part_of_the_country_you_live_in_now': ['Southern', 'Midwest', 'Mid-west']
    # Rationale: This feature directly captures whether a respondent explicitly uses "Midwest" (or a variation like "Mid-west", "Upper Midwest")
    # to describe where they currently live. This self-identification is a strong semantic indicator of their regional
    # belonging and perception, which is highly relevant for predicting their actual Census Region.
    midwest_terms_regex = r'(?i)midwest|mid-west|mid west|upper midwest|midewest'
    df_copy['self_reported_midwest_flag'] = df_copy['What_would_you_call_the_part_of_the_country_you_live_in_now'].str.contains(midwest_terms_regex, na=False).astype(int)

    # Define core and borderline Midwest states based on common geographical understanding
    core_midwest_states_cols = [
        'Do_you_consider_Illinois_state_as_part_of_the_Midwest',
        'Do_you_consider_Indiana_state_as_part_of_the_Midwest',
        'Do_you_consider_Iowa_state_as_part_of_the_Midwest',
        'Do_you_consider_Kansas_state_as_part_of_the_Midwest',
        'Do_you_consider_Michigan_state_as_part_of_the_Midwest',
        'Do_you_consider_Minnesota_state_as_part_of_the_Midwest',
        'Do_you_consider_Missouri_state_as_part_of_the_Midwest',
        'Do_you_consider_Nebraska_state_as_part_of_the_Midwest',
        'Do_you_consider_North_Dakota_state_as_part_of_the_Midwest',
        'Do_you_consider_Ohio_state_as_part_of_the_Midwest',
        'Do_you_consider_South_Dakota_state_as_part_of_the_Midwest',
        'Do_you_consider_Wisconsin_state_as_part_of_the_Midwest'
    ]

    borderline_midwest_states_cols = [
        'Do_you_consider_Arkansas_state_as_part_of_the_Midwest',
        'Do_you_consider_Colorado_state_as_part_of_the_Midwest',
        'Do_you_consider_Kentucky_state_as_part_of_the_Midwest',
        'Do_you_consider_Oklahoma_state_as_part_of_the_Midwest',
        'Do_you_consider_Pennsylvania_state_as_part_of_the_Midwest',
        'Do_you_consider_West_Virginia_state_as_part_of_the_Midwest',
        'Do_you_consider_Montana_state_as_part_of_the_Midwest',
        'Do_you_consider_Wyoming_state_as_part_of_the_Midwest'
    ]

    # Convert 'Yes'/'No' state columns to numeric (1/0) for calculation
    for col in core_midwest_states_cols + borderline_midwest_states_cols:
        df_copy[col + '_numeric'] = df_copy[col].map(yes_no_map)

    # (core_midwest_agreement_score: Sum of 'Yes' responses for generally accepted core Midwest states)
    # Input samples: 'Do_you_consider_Illinois_state_as_part_of_the_Midwest': ['No', 'Yes', 'Yes'], 'Do_you_consider_Indiana_state_as_part_of_the_Midwest': ['No', 'No', 'Yes'], ...
    # Rationale: This feature quantifies how many of the undisputed Midwestern states a respondent considers part of the Midwest.
    # It provides a more precise measure of their understanding of the core region, differentiating it from the
    # broader `midwest_state_count` which includes more ambiguous states. This can be highly indicative
    # of their own regional identity and actual location.
    df_copy['core_midwest_agreement_score'] = df_copy[[col + '_numeric' for col in c

In [7]:
display(Code(best_outcome.states.get("spatial_features")["generated_code"]))

def _sem_gen_features(df):
    return df